In [ ]:
import torch
import torch.nn.functional as F


In [ ]:
#find the peak location
def predict_argmax(sim_map):
    """
    sim_map: (H, W) similarity map
    returns: (x, y) pixel indices
    """
    h, w = sim_map.shape
    idx = torch.argmax(sim_map)
    y = idx // w
    x = idx % w
    return x.item(), y.item()


In [ ]:
def predict_window_soft_argmax(sim_map, window_size=5, beta=10.0):
    """
    sim_map: (H, W) similarity map
    window_size: odd number (e.g. 5)
    beta: temperature for softmax
    returns: (x, y) continuous coordinates
    """
    H, W = sim_map.shape

    # 1. Hard argmax
    max_idx = torch.argmax(sim_map)
    cy = max_idx // W
    cx = max_idx % W

    # 2. Define local window
    half = window_size // 2
    y1 = max(cy - half, 0)
    y2 = min(cy + half + 1, H)
    x1 = max(cx - half, 0)
    x2 = min(cx + half + 1, W)

    window = sim_map[y1:y2, x1:x2]

    # 3. Softmax inside window
    window_flat = window.reshape(-1) * beta
    probs = F.softmax(window_flat, dim=0)

    # 4. Expected coordinates
    ys, xs = torch.meshgrid(
        torch.arange(y1, y2, device=sim_map.device),
        torch.arange(x1, x2, device=sim_map.device),
        indexing="ij"
    )

    xs = xs.reshape(-1)
    ys = ys.reshape(-1)

    x = torch.sum(xs * probs)
    y = torch.sum(ys * probs)

    return x.item(), y.item()
